# Лабораторная работа 2. Обучение модели T5

**Задание**. Обучите модель `google/long-t5-tglobal-base` на датасете `SQUAD`. Сравните вручную на нескольких примерах качество ответов обученой модели с моделью `google-t5/t5-base`, а также на валидационной подвыборке датасета по метрикам `exact_match` и `f1`.

###Рекомендации:
- протестируйте код на небольшом подмножестве датасета,
- после отладки кода в финальном обучении скорее всего не получится использовать весь датасет, поэтому также используйте выборку из него,
- считается, что модели T5 (кроме t5-small) имеют проблемы с mixed precision training. При использовании `bf16=True` или `fp16=True` возникают `nan` в градиентах, поэтому не используйте смешанныую точность при обучении,
- не забудьте маскирование labels значением `-100`,
- используйте `report_to="none"` в аргументах обучения, чтобы не логировать процесс обучения в системе `wandb`,
- сохраняйте обученную модель, чтобы не запускать заново обучение

###Ниже следует "**примерная** заготовка" для вашего кода:

In [ ]:
!pip install evaluate

In [ ]:
import torch
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import load_dataset, DatasetDict
import numpy as np
from evaluate import load

In [ ]:
model_checkpoint = "google/long-t5-tglobal-base"
tokenizer = T5Tokenizer.from_pretrained(model_checkpoint)
model = T5ForConditionalGeneration.from_pretrained(model_checkpoint)

In [ ]:
def answer_question(question, context):
    """
    Генерация ответа на вопрос по контексту
    """
    input_text = f"question: {question} context: {context}"
    input_ids = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).input_ids

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    input_ids = input_ids.to(device)

    outputs = model.generate(
        input_ids,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

In [ ]:
test_context = """
The Amazon rainforest is a moist broadleaf tropical rainforest in the Amazon biome
that covers most of the Amazon basin of South America. The forest covers 5.5 million
square kilometers across nine countries.
"""
test_question = "How large is the Amazon rainforest?"

predicted_answer = answer_question(test_question, test_context)
print(f"\nТестовый пример:")
print(f"Вопрос: {test_question}")
print(f"Ответ модели: {predicted_answer}")

In [ ]:
dataset = load_dataset("squad")
print(f"Размер обучающей выборки: {len(dataset['train'])}")
print(f"Размер валидационной выборки: {len(dataset['validation'])}")

In [ ]:
# Создание подвыборки для обучения и валидации заданного размера

train_subset = dataset["train"].shuffle(seed=42).select(range(4000))
val_subset = dataset["validation"].shuffle(seed=42).select(range(200))
dataset = DatasetDict({
    "train": train_subset,
    "validation": val_subset
})
dataset

In [ ]:
def preprocess_squad_for_t5(examples):
    """
    Преобразование SQuAD в формат text-to-text для T5:
    Input: "question: <вопрос> context: <контекст>"
    Target: "<ответ>"
    """
    inputs = []
    targets = []

    for question, context, answers in zip(
        examples["question"],
        examples["context"],
        examples["answers"]
    ):
        input_text = f"question: {question} context: {context}"
        inputs.append(input_text)

        target_text = answers["text"][0] if len(answers["text"]) > 0 else ""
        targets.append(target_text)

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels["input_ids"] = [
        [(label if label != tokenizer.pad_token_id else -100) for label in labels_example]
        for labels_example in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
tokenized_dataset =  #ваш код

print(f"Пример токенизированных данных:")
print(tokenized_dataset["train"][0])

In [ ]:
data_collator = # ваш код

In [ ]:
squad_metric = load("squad")

def compute_metrics(eval_pred):
    """
    Вычисление метрик Exact Match и F1
    """
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.argmax(predictions, axis=-1)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    formatted_predictions = [
        {"id": str(i), "prediction_text": pred}
        for i, pred in enumerate(decoded_preds)
    ]
    formatted_references = [
        {"id": str(i), "answers": {"text": [label], "answer_start": [0]}}
        for i, label in enumerate(decoded_labels)
    ]

    results = squad_metric.compute(
        predictions=formatted_predictions,
        references=formatted_references
    )

    return {
        "exact_match": results["exact_match"],
        "f1": results["f1"]
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./t5-squad-results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=False,
    # добавьте/измените аргументы при необходимости
    report_to="none"
)

In [ ]:
trainer = #ваш код

In [ ]:
print("Начало обучения...")
trainer.train()

In [ ]:
print("\nОценка на валидационном наборе:")
results = trainer.evaluate()
print(f"Exact Match: {results['eval_exact_match']:.2f}")
print(f"F1 Score: {results['eval_f1']:.2f}")

In [ ]:
model.save_pretrained("./t5-squad-finetuned")
tokenizer.save_pretrained("./t5-squad-finetuned")
print("\nМодель сохранена в ./t5-squad-finetuned")

In [ ]:
test_context = """
The Amazon rainforest is a moist broadleaf tropical rainforest in the Amazon biome
that covers most of the Amazon basin of South America. The forest covers 5.5 million
square kilometers across nine countries.
"""
test_question = "How large is the Amazon rainforest?"

predicted_answer = answer_question(test_question, test_context)
print(f"\nТестовый пример:")
print(f"Вопрос: {test_question}")
print(f"Ответ модели: {predicted_answer}")

In [ ]:
#ваш код для сравнения с моделью google-t5/t5-base